# Раздел 4 · Механика векторного поиска: query_points, top-K, hnsw_ef

> Этот ноутбук продолжает Раздел 3 («Коллекции и точки») — использует уже созданную там коллекцию `BOOKS_COLLECTION`. Если ниже вылетит `AssertionError: Коллекция ... ещё не создана` — сначала откройте и выполните ноутбук Раздела 3, потом возвращайтесь сюда.

In [1]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


/home/dimon/vscode/examples_qdrant/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    SearchParams, RecommendQuery, RecommendInput
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [3]:
import os
from dotenv import load_dotenv
load_dotenv(".env")
client = QdrantClient(url="http://localhost:6336", api_key=os.environ["QDRANT__SERVICE__API_KEY"])

/tmp/ipykernel_67522/1300613524.py:4: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url="http://localhost:6336", api_key=os.environ["QDRANT__SERVICE__API_KEY"])


In [4]:
# CONTAINER_NAME = "qdrant_stepik"


# def sh(cmd):
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     return result.stdout.strip(), result.stderr.strip(), result.returncode


# sh("docker volume create qdrant_storage")

# _, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
# if exists != 0:
#     out, err, _ = sh(
#         "docker run -d --name {name} -p 6333:6336 -p 6334:6337 "
#         "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
#     )
# else:
#     out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
# print(out or err)

# client = QdrantClient(url="http://localhost:6336")
print(client.get_collections())


collections=[CollectionDescription(name='exercise1_books')]


In [5]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37290.20it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [6]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()
if client.collection_exists(BOOKS_COLLECTION):
    print("BOOKS_COLLECTION уже существует, точек:", client.count(BOOKS_COLLECTION).count)
else:
    client.create_collection(
        collection_name=BOOKS_COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )
    texts = [b["title"] + ". " + b["text"] for b in BOOKS]
    vectors = embed_passages(texts)
    points = [
        PointStruct(id=b["id"], vector=vec, payload={k: v for k, v in b.items() if k != "id"})
        for b, vec in zip(BOOKS, vectors)
    ]
    BATCH = 16
    for i in range(0, len(points), BATCH):
        client.upsert(collection_name=BOOKS_COLLECTION, points=points[i:i + BATCH])
    print("BOOKS_COLLECTION собрана заново, точек:", client.count(BOOKS_COLLECTION).count)

from qdrant_client.models import PointStruct
points = [
    PointStruct(
        id=b["id"],
        vector=embed_passages([b["title"] + ". " + b["text"]])[0],
        payload={k: v for k, v in b.items() if k != "id"},
    )
    for b in BOOKS
]
client.upsert(collection_name=BOOKS_COLLECTION, points=points)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29874.60it/s]


BOOKS_COLLECTION уже существует, точек: 87


UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [7]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Механика поиска

### Пример — семантический поиск

In [8]:
# query_points - основной метод поиска; query здесь - вектор запроса (уже с префиксом "query: " из embed_query)
example_hits = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("бетмен против супермена"),
    limit=5,
).points

for hit in example_hits:
    print(hit.id, round(hit.score, 3), hit.payload["genre"], "-", hit.payload["title"])


60 0.787 поэма - Руслан и Людмила
50 0.785 рассказ - Злоумышленник
10 0.782 роман - Преступление и наказание
66 0.778 поэма - Демон
40 0.778 повесть - Леди Макбет Мценского уезда


In [9]:
example_hits

[ScoredPoint(id=60, version=3, score=0.7866701, payload={'title': 'Руслан и Людмила', 'author': 'Александр Пушкин', 'year': 1820, 'genre': 'поэма', 'movement': 'романтизм', 'pages': 90, 'in_school_curriculum': True, 'text': 'В день свадьбы киевского князя Владимира его дочь Людмилу похищает злой карла-чародей Черномор, и убитый горем жених Руслан отправляется на её поиски вместе с тремя соперниками. По дороге Руслан сражается с великанской говорящей головой, получает волшебный меч и в финале побеждает Черномора, срезав его волшебную бороду, дарующую тому силу. Коварный соперник Фарлаф закалывает спящего Руслана и похищает спасённую Людмилу, но добрый волшебник Финн воскрешает героя живой водой.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=50, version=3, score=0.78459346, payload={'title': 'Злоумышленник', 'author': 'Антон Чехов', 'year': 1885, 'genre': 'рассказ', 'movement': 'реализм', 'pages': 4, 'in_school_curriculum': False, 'text': 'Тёмного крестьянина Дениса 

Реальный топ-1: `id=43`, "Толстый и тонкий" (Чехов), score `0.859` — ровно тот рассказ, где старый
товарищ мгновенно меняет тон, узнав чин собеседника.

## Задание 1

Найдите топ-5 по запросу **"мрачная история об убийстве и мучительных муках совести"**. Сохраните в `hits_ex1`.

**Используйте:** `client.query_points(...)`, `embed_query(...)`.

In [10]:
# ВАШ КОД ЗДЕСЬ
hits_ex1 = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("мрачная история об убийстве и мучительных муках совести"),
    limit=5,
).points

for hit in example_hits:
    print(hit.id, round(hit.score, 3), hit.payload["genre"], "-", hit.payload["title"])



60 0.787 поэма - Руслан и Людмила
50 0.785 рассказ - Злоумышленник
10 0.782 роман - Преступление и наказание
66 0.778 поэма - Демон
40 0.778 повесть - Леди Макбет Мценского уезда


In [11]:
def check_ex1(hits_ex1):
    assert len(hits_ex1) == 5, "ожидали top-5"
    assert hits_ex1[0].id == 10, (
        "первым результатом на запрос про убийство и муки совести ожидали id=10 "
        "(Преступление и наказание), получили id={}".format(hits_ex1[0].id)
    )
    print("OK: топ-1 —", hits_ex1[0].payload["title"], "score =", round(hits_ex1[0].score, 3))
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex1[0].id)


check_ex1(hits_ex1)


OK: топ-1 — Преступление и наказание score = 0.838
ОТВЕТ ДЛЯ STEPIK: 10


### Пример — recall: exact vs HNSW, один запрос

In [12]:
qv = embed_query("трагическая история о безответной любви")

# exact=True - честный полный перебор без графа HNSW, эталон для сравнения
exact = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(exact=True)).points
# hnsw_ef - ширина луча поиска по графу HNSW: больше = точнее, но медленнее
approx = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(hnsw_ef=64)).points

exact_ids = {p.id for p in exact}
approx_ids = {p.id for p in approx}
print("exact ids: ", exact_ids)
print("approx ids:", approx_ids)
print("recall@5 для этого запроса:", len(exact_ids & approx_ids) / 5)


exact ids:  {5, 38, 77, 78, 57}
approx ids: {5, 38, 77, 78, 57}
recall@5 для этого запроса: 1.0


## Задание 2

Повторите то же сравнение (exact vs `hnsw_ef`) для запроса **"лёгкий сатирический рассказ о чиновниках"**
и `hnsw_ef=16` (вместо 64). Сохраните id топ-5 в `exact_ids_ex2` и `approx_ids_ex2`.

**Используйте:** `client.query_points(...)`, `SearchParams(exact=True)`, `SearchParams(hnsw_ef=16)`.

In [22]:
# ВАШ КОД ЗДЕСЬ
qv = embed_query("лёгкий сатирический рассказ о чиновниках")
exact_ids_ex2 = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(exact=True)).points
approx_ids_ex2 = client.query_points(BOOKS_COLLECTION, query=qv, limit=5, search_params=SearchParams(hnsw_ef=16)).points
exact_ids_ex2 = [p.id for p in exact_ids_ex2]
approx_ids_ex2 = [p.id for p in approx_ids_ex2]


In [23]:
set(approx_ids_ex2)


{21, 23, 33, 43, 48}

In [24]:
def check_ex2(exact_ids_ex2, approx_ids_ex2):
    assert len(exact_ids_ex2) == 5 and len(approx_ids_ex2) == 5, "ожидали top-5 в обоих случаях"
    assert set(exact_ids_ex2) == set(approx_ids_ex2), (
        "на 87 точках exact и hnsw_ef=16 должны совпасть: {} vs {}".format(exact_ids_ex2, approx_ids_ex2)
    )
    print("OK: exact и hnsw_ef=16 совпали:", exact_ids_ex2)
    print("ОТВЕТ ДЛЯ STEPIK:", exact_ids_ex2[0])


check_ex2(exact_ids_ex2, approx_ids_ex2)


OK: exact и hnsw_ef=16 совпали: [43, 48, 23, 21, 33]
ОТВЕТ ДЛЯ STEPIK: 43


### Пример — group_by

In [26]:
# group_by схлопывает несколько хитов одного значения поля в одну группу - удобно для "не больше N на жанр/автора"
example_groups = client.query_points_groups(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("история о неудачной любви и расставании"),
    group_by="genre",
    group_size=3,   # не больше 2 хитов внутри одной группы
    limit=5,        # не больше 5 групп
)

for group in example_groups.groups:
    print(group.id, "->", [h.payload["title"] for h in group.hits])


роман -> ['Обыкновенная история', 'Что делать?', 'Униженные и оскорблённые']
пьеса -> ['Лес', 'Бесприданница', 'Горе от ума']
повесть -> ['Ася', 'Крейцерова соната', 'Первая любовь']
рассказ -> ['После бала', 'Старуха Изергиль', 'Ионыч']
поэма -> ['Демон', 'Кавказский пленник', 'Песня про купца Калашникова']


## Задание 3

То же самое, но для запроса **"история о разочаровании в жизни и внутренней пустоте"** — сохраните в `groups_ex3`.

**Используйте:** `client.query_points_groups(...)`, `embed_query(...)`.

In [27]:
# ВАШ КОД ЗДЕСЬ
groups_ex3 = client.query_points_groups(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("история о разочаровании в жизни и внутренней пустоте"),
    group_by="genre",
    group_size=2,
    limit=5
)


In [31]:
groups_ex3

GroupsResult(groups=[PointGroup(hits=[ScoredPoint(id=58, version=3, score=0.8245394, payload={'title': 'Старуха Изергиль', 'author': 'Максим Горький', 'year': 1894, 'genre': 'рассказ', 'movement': 'реализм', 'pages': 20, 'in_school_curriculum': True, 'text': 'Старая цыганка Изергиль рассказывает попутчику две легенды и историю собственной бурной жизни. Первая легенда — о гордом Ларре, сыне орла, за высокомерие и презрение к людям обречённом на вечное одинокое бессмертие. Вторая — о смелом юноше Данко, который вырвал из груди своё пылающее сердце и, освещая им путь, вывел соплеменников из тёмного леса, за что был растоптан неблагодарной толпой сразу после спасения.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=44, version=3, score=0.8127705, payload={'title': 'Человек в футляре', 'author': 'Антон Чехов', 'year': 1898, 'genre': 'рассказ', 'movement': 'реализм', 'pages': 15, 'in_school_curriculum': True, 'text': 'Учитель греческого языка Беликов боится всего нового и ж

In [32]:
for group in groups_ex3.groups:
    print(group.id, "->", [h.payload["title"] for h in group.hits])


рассказ -> ['Старуха Изергиль', 'Человек в футляре']
роман -> ['Обыкновенная история', 'Униженные и оскорблённые']
повесть -> ['Записки сумасшедшего', 'Крейцерова соната']
пьеса -> ['Горе от ума', 'Гроза']
поэма -> ['Кому на Руси жить хорошо', 'Мцыри']


In [33]:
def check_ex3(groups_ex3):
    assert len(groups_ex3.groups) <= 5, "ожидали не больше 5 групп (limit=5)"
    for group in groups_ex3.groups:
        assert len(group.hits) <= 2, "group_size=2, а в группе {} оказалось {} хитов".format(group.id, len(group.hits))
    print("OK:", [(g.id, len(g.hits)) for g in groups_ex3.groups])
    print("ОТВЕТ ДЛЯ STEPIK:", len(groups_ex3.groups))


check_ex3(groups_ex3)


OK: [('рассказ', 2), ('роман', 2), ('повесть', 2), ('пьеса', 2), ('поэма', 2)]
ОТВЕТ ДЛЯ STEPIK: 5


### Пример — Recommend API

`positive` — id трёх тяжёлых, тёмных романов Достоевского, `negative` — id лёгкого юмористического рассказа Чехова. Признак успеха здесь не "результат — обязательно Достоевский" (Recommend усредняет смысл, а не автора), а то, что среди результатов не окажется лёгких юмористических вещей из того же "негативного" кластера:

In [34]:
LIGHT_COMEDY_IDS = {42, 43, 48, 49, 50}  # короткие сатирические рассказы Чехова

# RecommendInput усредняет векторы positive и отталкивается от векторов negative - это не "похоже на A, но не на B"
# буквально, а векторная арифметика поверх эмбеддингов
example_recs = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=RecommendQuery(
        recommend=RecommendInput(positive=[10, 12, 13], negative=[42]),
    ),
    limit=5,
).points

for h in example_recs:
    print(h.id, h.payload["genre"], "-", h.payload["title"])

result_ids = {h.id for h in example_recs}
print("пересечение с лёгкой комедией:", result_ids & LIGHT_COMEDY_IDS)


6 роман - Отцы и дети
17 роман - Воскресение
19 роман - Что делать?
28 повесть - Дубровский
14 роман - Униженные и оскорблённые
пересечение с лёгкой комедией: set()


Реальный результат: `[6, 17, 19, 28, 14]` — Отцы и дети, Воскресение, Что делать?, Дубровский,
Униженные и оскорблённые. Ни одного результата из набора лёгких сатирических рассказов Чехова —
направление `positive`/`negative` честно увело поиск от лёгкого тона, хотя и не свело всё строго
к одному автору.

## Задание 4

Разверните направление: `positive` = id трёх лёгких сатирических рассказов Чехова (`42`, `43`, `48`),
`negative` = id тяжёлых романов Достоевского (`{10, 11, 12, 13}`). Как и в примере выше, признак успеха —
отсутствие в топ-5 книг из «негативного» кластера: здесь это любой из четырёх романов Достоевского
в датасете, а не только сам id `10`. Сохраните топ-5 в `hits_ex4`.

**Используйте:** `client.query_points(...)`, `RecommendQuery(...)`, `RecommendInput(positive=..., negative=...)`.

In [40]:
# ВАШ КОД ЗДЕСЬ
positive = [42, 43, 48] 
negative = [10, 11, 12, 13] 

hits_ex4 = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=RecommendQuery(
        recommend=RecommendInput(positive=positive, negative=negative),
    ),
    limit=5,
).points

for h in hits_ex4:
    print(h.id, h.payload["genre"], "-", h.payload["title"])

result_ids = {h.id for h in hits_ex4}
print("пересечение с лёгкой комедией:", set(positive) & set(negative))



74 пьеса - Ревизор
49 рассказ - Унтер Пришибеев
21 повесть - Шинель
55 рассказ - После бала
23 повесть - Записки сумасшедшего
пересечение с лёгкой комедией: set()


In [41]:
HEAVY_TRAGEDY_IDS = {10, 11, 12, 13}  # все тяжёлые романы Достоевского в датасете


def check_ex4(hits_ex4):
    assert len(hits_ex4) == 5
    result_ids = {h.id for h in hits_ex4}
    overlap = result_ids & HEAVY_TRAGEDY_IDS
    assert not overlap, "среди результатов не должно быть тяжёлых трагических романов, нашли {}".format(overlap)
    print("OK:", [(h.id, h.payload["genre"], h.payload["title"]) for h in hits_ex4])
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex4[0].id)


check_ex4(hits_ex4)


OK: [(74, 'пьеса', 'Ревизор'), (49, 'рассказ', 'Унтер Пришибеев'), (21, 'повесть', 'Шинель'), (55, 'рассказ', 'После бала'), (23, 'повесть', 'Записки сумасшедшего')]
ОТВЕТ ДЛЯ STEPIK: 74
